In [24]:
import pandas as pd

# Load datasets, skipping the metadata block
df_2023 = pd.read_csv(
    'midas-open_uk-mean-wind-obs_dv-202507_avon_62122_almondsbury_qcv-1_2023.csv',
    skiprows=82
)

df_2024 = pd.read_csv(
    'midas-open_uk-mean-wind-obs_dv-202507_avon_62122_almondsbury_qcv-1_2024.csv',
    skiprows=82
)

In [25]:
df_2023.head()
df_2023.columns

Index(['ob_end_time', 'id_type', 'id', 'ob_hour_count', 'met_domain_name',
       'version_num', 'src_id', 'rec_st_ind', 'mean_wind_dir',
       'mean_wind_speed', 'max_gust_dir', 'max_gust_speed', 'max_gust_ctime',
       'mean_wind_dir_q', 'mean_wind_speed_q', 'max_gust_dir_q',
       'max_gust_speed_q', 'max_gust_ctime_q', 'mean_wind_dir_j',
       'mean_wind_speed_j', 'max_gust_dir_j', 'max_gust_speed_j',
       'meto_stmp_time', 'midas_stmp_etime'],
      dtype='object')

In [26]:
cols = ['ob_end_time', 'mean_wind_speed', 'mean_wind_dir']
df_2023 = df_2023[cols]
df_2024 =df_2024[cols]

df_2024.head(10)
df_2023.head(10)

,ob_end_time,mean_wind_speed,mean_wind_dir
0,2023-01-01 00:00:00,11.0,210.0
1,2023-01-01 01:00:00,11.0,210.0
2,2023-01-01 02:00:00,10.0,210.0
3,2023-01-01 03:00:00,9.0,200.0
4,2023-01-01 04:00:00,8.0,190.0
5,2023-01-01 05:00:00,10.0,200.0
6,2023-01-01 06:00:00,9.0,190.0
7,2023-01-01 07:00:00,12.0,210.0
8,2023-01-01 08:00:00,9.0,200.0
9,2023-01-01 09:00:00,8.0,190.0


In [27]:
# Missing values count
print("2023 missing values:")
print(df_2023.isnull().sum())

print("\n2024 missing values:")
print(df_2024.isnull().sum())

2023 missing values:
ob_end_time        0
mean_wind_speed    3
mean_wind_dir      3
dtype: int64

2024 missing values:
ob_end_time        0
mean_wind_speed    2
mean_wind_dir      2
dtype: int64


In [28]:
# 2023 missing rows
df_2023[df_2023.isnull().any(axis=1)]

,ob_end_time,mean_wind_speed,mean_wind_dir
5453,2023-08-16 14:00:00,NaN,NaN
7818,2023-11-23 03:00:00,NaN,NaN
8751,end data,NaN,NaN


In [29]:
# 2024 missing rows
df_2024[df_2024.isnull().any(axis=1)]

,ob_end_time,mean_wind_speed,mean_wind_dir
1213,2024-02-20 18:00:00,NaN,NaN
8774,end data,NaN,NaN


In [30]:
# Remove rows where ob_end_time is not a valid timestamp
df_2023 = df_2023[df_2023['ob_end_time'] != 'end data']
df_2024 = df_2024[df_2024['ob_end_time'] != 'end data']

In [31]:
#convert to datetime
df_2023['ob_end_time'] = pd.to_datetime(df_2023['ob_end_time'])
df_2024['ob_end_time'] = pd.to_datetime(df_2024['ob_end_time'])

In [32]:
#sort values
df_2023 = df_2023.sort_values('ob_end_time')
df_2024 = df_2024.sort_values('ob_end_time')

#index the data
df_2023 = df_2023.set_index('ob_end_time')
df_2024 = df_2024.set_index('ob_end_time')

#interpolate the values (taking the average in between the row above and below)
df_2023 = df_2023.interpolate(method='time')
df_2024 = df_2024.interpolate(method='time')

#reset index
df_2023 = df_2023.reset_index()
df_2024 = df_2024.reset_index()

#check worked
print(df_2023.isnull().sum())
print(df_2024.isnull().sum())

df_2023.loc[df_2023['ob_end_time'] == '2023-08-16 14:00:00']
df_2023.loc[5452:5454]  # row before, interpolated, row after

ob_end_time        0
mean_wind_speed    0
mean_wind_dir      0
dtype: int64
ob_end_time        0
mean_wind_speed    0
mean_wind_dir      0
dtype: int64


,ob_end_time,mean_wind_speed,mean_wind_dir
5452,2023-08-16 10:00:00,4.0,20.0
5453,2023-08-16 14:00:00,5.6,20.0
5454,2023-08-16 15:00:00,6.0,20.0


In [33]:
# Calculate time differences
df_2023['time_diff'] = df_2023['ob_end_time'].diff()
df_2024['time_diff'] = df_2024['ob_end_time'].diff()

print("2023 time difference counts:")
print(df_2023['time_diff'].value_counts().sort_index())

print("\n2024 time difference counts:")
print(df_2024['time_diff'].value_counts().sort_index())

2023 time difference counts:
time_diff
0 days 01:00:00    8748
0 days 04:00:00       1
0 days 07:00:00       1
Name: count, dtype: int64

2024 time difference counts:
time_diff
0 days 01:00:00    8771
0 days 06:00:00       2
Name: count, dtype: int64


In [34]:
# Rows where the gap is not 1 hour (without the first row)
non_hourly_2023 = df_2023[
    (df_2023['time_diff'].notna()) & (df_2023['time_diff'] != pd.Timedelta(hours=1))
]
non_hourly_2024 = df_2024[
    (df_2024['time_diff'].notna()) & (df_2024['time_diff'] != pd.Timedelta(hours=1))
]

print("2023 non-hourly gaps:")
print(non_hourly_2023[['ob_end_time', 'time_diff']].head(20))

print("\n2024 non-hourly gaps:")
print(non_hourly_2024[['ob_end_time', 'time_diff']].head(20))

2023 non-hourly gaps:
             ob_end_time       time_diff
1571 2023-03-07 17:00:00 0 days 07:00:00
5453 2023-08-16 14:00:00 0 days 04:00:00

2024 non-hourly gaps:
             ob_end_time       time_diff
1213 2024-02-20 18:00:00 0 days 06:00:00
5598 2024-08-21 16:00:00 0 days 06:00:00
